In [4]:
!pip install streamlit==1.32.0 pyngrok --quiet

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

MODEL_PATH = "/content/drive/MyDrive/unet_task4_fast_final.pth"
print("Model path:", MODEL_PATH)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 68.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 85.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 96.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 17.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
db-dtypes 1.4.4 requires packaging>=24.2.0, but you have packaging 23.2 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.12.0.88 requires nump

In [5]:
%%writefile app.py
import streamlit as st
import torch
import torch.nn as nn
import cv2
import numpy as np

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.c = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, 1, 1), nn.ReLU(),
            nn.Conv2d(out_ch, out_ch, 3, 1, 1), nn.ReLU()
        )
    def forward(self, x): return self.c(x)

class UNetLite(nn.Module):
    def __init__(self):
        super().__init__()
        self.d1 = DoubleConv(3,32)
        self.d2 = DoubleConv(32,64)
        self.d3 = DoubleConv(64,128)
        self.bott = DoubleConv(128,256)

        self.up3 = nn.ConvTranspose2d(256,128,2,2)
        self.u3  = DoubleConv(256,128)

        self.up2 = nn.ConvTranspose2d(128,64,2,2)
        self.u2  = DoubleConv(128,64)

        self.up1 = nn.ConvTranspose2d(64,32,2,2)
        self.u1  = DoubleConv(64,32)

        self.out = nn.Conv2d(32,1,1)

    def forward(self,x):
        c1 = self.d1(x); p1 = nn.MaxPool2d(2)(c1)
        c2 = self.d2(p1); p2 = nn.MaxPool2d(2)(c2)
        c3 = self.d3(p2); p3 = nn.MaxPool2d(2)(c3)
        c4 = self.bott(p3)
        u3 = self.up3(c4); u3 = self.u3(torch.cat([u3,c3],1))
        u2 = self.up2(u3); u2 = self.u2(torch.cat([u2,c2],1))
        u1 = self.up1(u2); u1 = self.u1(torch.cat([u1,c1],1))
        return self.out(u1)

MODEL_PATH = "/content/drive/MyDrive/unet_task4_fast_final.pth"
model = UNetLite()
model.load_state_dict(torch.load(MODEL_PATH, map_location="cpu"))
model.eval()

st.title("Task 6 - UNet Segmentation")
file = st.file_uploader("Upload Image", type=["jpg","jpeg","png"])

if file:
    img_bytes = np.frombuffer(file.read(), np.uint8)
    img = cv2.imdecode(img_bytes, cv2.IMREAD_COLOR)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    resized = cv2.resize(img, (128,128))

    st.image(resized, caption="Uploaded Image")

    t = torch.tensor(resized/255., dtype=torch.float32).permute(2,0,1).unsqueeze(0)
    with torch.no_grad():
        pred = torch.sigmoid(model(t))[0,0].numpy()

    mask = (pred > 0.5).astype(np.uint8) * 255
    st.image(mask, caption="Predicted Mask", clamp=True)


Writing app.py


In [7]:
!kill -9 $(lsof -t -i:8501)
!pkill streamlit


kill: usage: kill [-s sigspec | -n signum | -sigspec] pid | jobspec ... or kill -l [sigspec]


In [9]:
%%writefile app.py
import streamlit as st
import torch
import torch.nn as nn
import cv2
import numpy as np
from PIL import Image

MODEL_PATH = "/content/drive/MyDrive/unet_task4_fast_final.pth"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ----- Model -----
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.c = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, 1, 1), nn.ReLU(),
            nn.Conv2d(out_ch, out_ch, 3, 1, 1), nn.ReLU()
        )
    def forward(self, x): return self.c(x)

class UNetLite(nn.Module):
    def __init__(self):
        super().__init__()
        self.d1 = DoubleConv(3,32)
        self.d2 = DoubleConv(32,64)
        self.d3 = DoubleConv(64,128)
        self.bott = DoubleConv(128,256)

        self.up3 = nn.ConvTranspose2d(256,128,2,2)
        self.u3  = DoubleConv(256,128)

        self.up2 = nn.ConvTranspose2d(128,64,2,2)
        self.u2  = DoubleConv(128,64)

        self.up1 = nn.ConvTranspose2d(64,32,2,2)
        self.u1  = DoubleConv(64,32)

        self.out = nn.Conv2d(32,1,1)

    def forward(self,x):
        c1 = self.d1(x); p1 = nn.MaxPool2d(2)(c1)
        c2 = self.d2(p1); p2 = nn.MaxPool2d(2)(c2)
        c3 = self.d3(p2); p3 = nn.MaxPool2d(2)(c3)
        c4 = self.bott(p3)
        u3 = self.up3(c4); u3 = self.u3(torch.cat([u3,c3],1))
        u2 = self.up2(u3); u2 = self.u2(torch.cat([u2,c2],1))
        u1 = self.up1(u2); u1 = self.u1(torch.cat([u1,c1],1))
        return self.out(u1)

# Load Model
model = UNetLite().to(DEVICE)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()

st.title("Object Segmentation App (Task 6)")
st.write("Upload an image to get the predicted mask.")

uploaded = st.file_uploader("Choose an image", type=["jpg","png","jpeg"])

if uploaded:
    img = Image.open(uploaded).convert("RGB")
    img_np = np.array(img)
    img_resized = cv2.resize(img_np, (128,128))
    t = torch.tensor(img_resized/255., dtype=torch.float32).permute(2,0,1).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        pred = torch.sigmoid(model(t))[0,0].cpu().numpy()

    pred_bin = (pred > 0.5).astype("uint8") * 255

    st.image(img_np, caption="Original Image")
    st.image(pred_bin, caption="Predicted Mask", clamp=True)


Overwriting app.py


In [11]:
!pip install pyngrok

from pyngrok import ngrok

# Shivang's token (Tumne diya tha)
ngrok.set_auth_token("36T8q53H7RM7g6HH3WZQ5AP1Mrv_5M5AgvXEtKN9CymZfWCs3")

print("NGROK TOKEN SET SUCCESSFULLY!")


NGROK TOKEN SET SUCCESSFULLY!


In [12]:
# Run Streamlit app in background
!streamlit run app.py --server.port 8501 &> /dev/null &

# Create public ngrok URL
public_url = ngrok.connect(8501)
public_url


<NgrokTunnel: "https://polygalaceous-superwrought-willetta.ngrok-free.dev" -> "http://localhost:8501">